<a href="https://colab.research.google.com/github/yeogyung/welfare-resource-finder/blob/main/20%EA%B8%B0_%EB%8F%84%EC%A0%84%ED%95%99%EA%B8%B0%EC%A0%9C_%EB%B0%95%EC%97%AC%EA%B2%BD_RAG_%ED%94%84%EB%A1%9C%ED%86%A0%ED%83%80%EC%9E%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 패키지 설치

In [ ]:
!pip install transformers accelerate bitsandbytes sentence-transformers faiss-cpu pandas openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 92.2 MB/s eta 0:00:00


2. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [ ]:
from transformers import BitsAndBytesConfig, AutoConfig

3. 노인복지 자원DB_엑셀 파일 업로드

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 노인복지자원DB_MVP_v1.xlsx to 노인복지자원DB_MVP_v1.xlsx


3-1. 시트명 확인

In [ ]:
excel_file = pd.ExcelFile("노인복지자원DB_MVP_v1.xlsx")
print(excel_file.sheet_names)

['노인복지DB_v1', '현황 요약']


4. DB 로드 + 검색 텍스트 생성

In [ ]:
# 1) 엑셀 로드
df = pd.read_excel(
    "노인복지자원DB_MVP_v1.xlsx",
    sheet_name="노인복지DB_v1"
)

print(f"DB 로드 완료: {len(df)}건")
print(df[['ID', '구분', '카테고리', '서비스명']].head(10))

# 2) 검색용 텍스트 / 노인 친화 설명 텍스트 생성
def make_search_text(row):
    return (
        f"서비스명: {row['서비스명']} | "
        f"카테고리: {row['카테고리']} | "
        f"지원대상: {row['지원 대상']} | "
        f"설명: {row['서비스 요약 (노인 친화)']}"
    )

def make_friendly_text(row):
    contact = (
        f" 문의: {row['문의처']}"
        if pd.notna(row['문의처']) and str(row['문의처']).strip()
        else ""
    )
    return (
        f"[{row['서비스명']}]\n"
        f"{row['서비스 요약 (노인 친화)']}\n"
        f"대상: {row['지원 대상']}{contact}"
    )

df['search_text'] = df.apply(make_search_text, axis=1)
df['friendly_text'] = df.apply(make_friendly_text, axis=1)
print("검색 텍스트 생성 완료")

DB 로드 완료: 82건
   ID  구분   카테고리                서비스명
0   1  공공   기타지원          노인복지민간단체지원
1   2  공공   경제지원    노인장기요양보험 복지용구 급여
2   3  공공  돌봄서비스    산재근로자 합병증 등 예방관리
3   4  공공   경제지원       특별현금급여(가족요양비)
4   5  공공   경제지원    요양급여(보조기)-산재보험급여
5   6  공공  돌봄서비스  독거노인·장애인 응급안전안심서비스
6   7  공공   사회참여          큰글자책 보급 지원
7   8  공공   경제지원          주택담보노후연금보증
8   9  공공   사회참여       취약지역 어르신 문화누림
9  10  공공   경제지원       노인 안검진 및 개안수술
검색 텍스트 생성 완료


6. 임베딩 + FAISS

In [ ]:
print("임베딩 모델 로드 중... (1~2분)")
embed_model = SentenceTransformer("jhgan/ko-sroberta-multitask")

print("임베딩 생성 중... (1~2분)")
embeddings = embed_model.encode(
    df['search_text'].tolist(),
    show_progress_bar=True,
    batch_size=32
)
embeddings = np.array(embeddings).astype('float32')

faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"FAISS 인덱스 구축 완료 ({index.ntotal}건)")

임베딩 모델 로드 중... (1~2분)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

임베딩 생성 중... (1~2분)


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

FAISS 인덱스 구축 완료 (82건)


7. 검색 함수

In [ ]:
def search_welfare(query, top_k=3):
    query_vec = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = df.iloc[idx]
        results.append({
            'name': row['서비스명'],
            'category': row['카테고리'],
            'friendly_text': row['friendly_text'],
            'score': float(score)
        })
    return results

print("=== 검색 테스트 ===")
for r in search_welfare("혼자 사는데 쓰러지면"):
    print(f"[유사도 {r['score']:.3f}] {r['name']} ({r['category']})")

=== 검색 테스트 ===
[유사도 0.358] 독거노인 생활필수품 지원 (기타지원)
[유사도 0.284] 독거노인·장애인 응급안전안심서비스 (돌봄서비스)
[유사도 0.280] 재가급여 (경제지원)


8. LLM(EEVE) 로드

In [ ]:
MODEL_ID = "yanolja/EEVE-Korean-Instruct-2.8B-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
config.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=600,
    temperature=0.7,
    do_sample=True,
    repetition_penalty=1.1
)

print("EEVE 모델 로드 완료")

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


EEVE 모델 로드 완료


9. RAG 추천 함수

In [ ]:
# 노인 친화적 답변을 위한 시스템 가이드라인 설정
SYSTEM_BASE = """당신은 한국 노인이 복지 자원을 쉽게 찾을 수 있도록 도와주는 친절한 안내 도우미입니다.
[규칙]
1. 아래 제공된 복지 자원 목록에서만 답변합니다.
2. 전문 용어 대신 쉬운 말을 씁니다. (예: "재가급여" -> "집으로 와서 도와드리는 서비스")
3. 각 자원을 2~3문장으로 설명하고, 전화번호나 신청 방법을 꼭 포함합니다.
4. 목록에 없는 정보는 "주민센터나 129에 확인해 보세요"로 안내합니다.
5. 따뜻하고 존댓말로 대화합니다.
"""

def rag_recommend(user_input, top_k=3):

    retrieved = search_welfare(user_input, top_k=top_k)

    resource_text = "\n\n".join([r['friendly_text'] for r in retrieved])
    system_prompt = f"{SYSTEM_BASE}\n\n[검색된 관련 복지 자원]\n{resource_text}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = pipe(prompt)[0]["generated_text"]
    response = output[len(prompt):].strip()

    return {
        "question": user_input,
        "retrieved": [r['name'] for r in retrieved],
        "answer": response
    }

print("RAG 추천 시스템 준비 완료")

RAG 추천 시스템 준비 완료


* RAG Test 1

In [ ]:
result = rag_recommend("혼자 사는데 갑자기 쓰러지면 어떡하죠?")
print(f"📌 검색된 자원: {', '.join(result['retrieved'])}")
print(f"\n💬 답변:\n{result['answer']}")

Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📌 검색된 자원: 독거노인 생활필수품 지원, 독거노인·장애인 응급안전안심서비스, 노인맞춤돌봄서비스

💬 답변:
우울증이나 우울증 위험이 높고 도움을 필요로 하는 분들을 위해 마련된 '재가급여'라고 불리는 지원을 소개합니다. 이 서비스는 직접 방문하지 않고도 도움이 필요한 이들을 돕기 위한 것입니다.

재가급여는 65세 이상의 독거노인들이 겪어야 할 여러 어려움들을 해결해 주며, 그들의 일상생활을 향상시켜주죠. 물론, 건강 문제가 있거나 특별한 요구사항이 있는 경우에는 지역 내 거주민센터에 문의하거나 해당 기관과 상담 후 필요한 모든 지원을 받을 수 있습니다.

지금부터 가입하기 시작해서 재가급여의 다양한 혜택을 경험해보세요! 핸드폰 번호로 연락해주세요(전화번호 010) 또는 이메일로 연락해 주시면 도움을 드리겠습니다. 감사합니다. 

재가급여를 받으려면 다음 단계를 따르세요:

1. **등록**: 주민등록증과 여권 사진을 지참하세요.
2. **신청**: 신청서를 작성하여 우편(지방자치단체의 추천 발송지 지정 주소)에 보내주세요.
3. **추천**: 필요한 경우, 추천인을 통해 추천서를 발송받으실 수도 있으니 참고하시기 바랍니다.
4. **유료**: 신청받은 후 보험가입료를 지불해야 합니다.

재가급여에서 제공하는 것은 다음과 같습니다:

* **생활필수물품**: 가구용품, 옷감, 주방용품, 화장품 등 필수품을 구성하실 수 있습니다.
* **숙박비용**: 개인 숙소를 찾아주는 기회와 필요할 경우 요금제 할인 등 추가적인 도움을 줄 수 있으며, 이는 특정 시기에 제한되어 있을 수 있기 때문에 미리 확인해 주시기 바랍니다.
* **간호**: 맞벌이 부부나 바쁜 업무로 인해 혼자서 관리할 수 없는 신체적인 필요를 뒷받침해주는 서비스로, 예를 들어 식사 준비, 청결 관리, 생리주기 등이 포함됩니다.

재가급여의 다양한 혜택을 살펴보며, 귀하에게 가장 적합한 버전을 선택해서 활용하시길 바랍니다. 궁금한 사항이 있으신가요? 아니면 지금 당장 그 혜택을 받으시기를 기다리시는

* RAG Test 2

In [ ]:
result = rag_recommend("매달 생활비가 부족해요. 집은 있어요.")
print(f"📌 검색된 자원: {', '.join(result['retrieved'])}")
print(f"\n💬 답변:\n{result['answer']}")

* RAG Test 3

In [ ]:
result = rag_recommend("뭔가 배우거나 활동하고 싶은데 어디 가면 돼요?")
print(f"📌 검색된 자원: {', '.join(result['retrieved'])}")
print(f"\n💬 답변:\n{result['answer']}")

9.2. RAG 성능 개선안_v2 (Strict Prompting & Few-shot 적용)

In [ ]:
# Few-shot 및 출력 제약 조건 강화
IMPROVED_SYSTEM_BASE = """당신은 어르신을 위한 복지 안내원입니다.
반드시 아래 제공된 [검색된 복지 자원]의 정보만 사용하여 다음 양식으로 답변하세요.

[답변 양식]
1. 인사말: (상황에 공감하는 짧은 인사)
2. 추천 서비스: (서비스명)
3. 상세 내용: (어르신이 이해하기 쉬운 2~3문장 설명)
4. 신청 방법: (지원 대상 및 문의처 정보)

[주의사항]
- 제공된 정보에 없는 전화번호나 신청 서류(여권 등)는 절대 지어내지 마세요.
- 모르는 내용은 "주민센터(129)에 문의하세요"라고 안내하세요.

"""

def rag_recommend_v2(user_input, top_k=2):
    retrieved = search_welfare(user_input, top_k=top_k)

    resource_text = "\n\n".join([r['friendly_text'] for r in retrieved])

    system_prompt = f"{IMPROVED_SYSTEM_BASE}\n\n[검색된 복지 자원]\n{resource_text}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    output = pipe(prompt)[0]["generated_text"]
    response = output[len(prompt):].strip()

    return response

print("=== [개선안] RAG v2 테스트 실행 ===")
test_query = "혼자 사는데 갑자기 쓰러지면 어떡하죠?"
print(f"질문: {test_query}")
print(f"답변:\n{rag_recommend_v2(test_query)}")

Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [개선안] RAG v2 테스트 실행 ===
질문: 혼자 사는데 갑자기 쓰러지면 어떡하죠?
답변:
사소한 실수에도 불구하고, 혼자 방치되었을 때 발생하는 건강상의 위험과 안타까움을 이해하는 것이 중요합니다. 이러한 문제를 해결하기 위해서는 중앙정부와 지방 정부가 제공하는 다양한 복지 자원이 필요합니다. 이 목록에는 65세 이상의 독거노인들이 안전하고 편안한 생활을 위해 필수적인 물품을 제공하는 '생활필수품 세트'가 포함되어 있습니다. 또한, 신체적 또는 정신적으로 취약한 노인이나 장애인들이 응급 상황에 직면했을 때 도움을 줄 수 있는 '응급안심서비스'도 있습니다. 이 도움들은 여러분의 생계를 책임지고 안전하게 지키는 데 도움을 줄 것입니다. 이 정보는 당신과 당신의 가족을 보호하는데 필수적입니다.

[민간 복지 자원]
[다문화 거주자의 영리 의료 지원 프로그램](의료복지사업): 다문화 거주자들이 건강한 생활 방식을 유지하기 위해 전문적이고 전문적인 지원을 받을 수 있게 도와주는 사업으로, 이는 다양한 문화적 배경이 공존하는 환경에서 의료 문제에 효과적으로 대응하기 위해 필요한 모든 세부 사항을 포함합니다.
대상: 다문화 거주자들의 참여: 국립 보건원 상담소 02-3480-1209, 네이버 페이북 [다문화 의료 지원 프로그램](https://naver.me/davidhealive)

[아동수당][기타 아동 지원 비용]: 아동에게 중요한 교육 및 여가 활동 기회를 보장하기 위해 지급되는 대규모 현금 지급 계획으로서의 아동수당입니다.
대상: 양육자 상담: 사회보장청 02-2100-1000 청소년교육복지국 02-2100-1030

[취약계층 주거 환경 개선](주택환경개선사업): 주거 환경이 열악한 취약계층 가정에 직접 방문해 에너지 효율적인 교체, 수리, 조립 기술 교육을 제공하여 에너지 비용을 줄이고 더 잘 살고자 하는 사람들의 삶에 대한 중요성을 강조합니다.
대상: 주택건설업체 문의: 국가주택공사 03-3600-0000, 전국 아파트 공급업협회 044-203

>> 답변에 환각이 심각한 수준

9.3. RAG 성능 개선안_v3 (Strict Constraints & url 포함)

In [ ]:
# 가독성 개선, 근거 강화_url 포함
def rag_recommend_v3(user_input, top_k=2):

    retrieved = search_welfare(user_input, top_k=top_k)

    final_output = []

    for i, r in enumerate(retrieved):
        row_data = df[df['서비스명'] == r['name']].iloc[0]

        url_val = row_data.iloc[12] if len(row_data) > 12 else "주민센터 문의"

        if pd.isna(url_val) or str(url_val).strip() == "":
            url_val = "공식 홈페이지 확인 필요 (129 문의)"

        detail_info = r['friendly_text'].split('대상:')[0].replace('['+r['name']+']', '').strip()

        item_template = f"""
[{i+1}] {r['name']} ({r['category']})
▶ 서비스 설명: {detail_info}
▶ 누가 받을 수 있나요?: {r['friendly_text'].split('대상:')[-1].strip()}
▶ 상세 확인 및 신청(URL): {url_val}
        """
        final_output.append(item_template)

    STRICT_PROMPT = f"""당신은 노인복지 전문가입니다.
어르신이 "{user_input}"라고 물어보셨습니다.
어르신의 마음을 위로하고 안심시켜드리는 따뜻한 첫 인사말을 한 문장으로만 다정하게 작성하세요.
성별을 특정하지 말고, 반말을 쓰지 마세요. '어르신'이라는 명칭으로 답을 하세요.
'신뢰도'나 '답변 양식' 같은 단어는 절대 쓰지 마세요.

인사말:"""

    output = pipe(STRICT_PROMPT, max_new_tokens=100, temperature=0.1, do_sample=False)[0]["generated_text"]

    llm_intro = output.split("인사말:")[-1].strip()

    llm_intro = llm_intro.split('\n')[0].replace("(인사말 입력)", "").strip()

    full_response = f"💬 {llm_intro}\n" + "\n".join(final_output)

    return full_response

# 테스트 실행
print("=== [개선안] RAG v3 테스트 실행 ===")
print(rag_recommend_v3("혼자 사는데 갑자기 쓰러지면 어떡하죠?"))

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [개선안] RAG v3 테스트 실행 ===
💬 "안녕하세요, 어르신님! 저는 노인 복지 전문가입니다. 제가 도와드릴 수 있는 모든 것을 알려드리겠습니다."

[1] 독거노인 생활필수품 지원 (기타지원)
▶ 서비스 설명: 생활필수품 세트
▶ 누가 받을 수 있나요?: 65세 이상 수급자 독거노인
▶ 상세 확인 및 신청(URL): 공식 홈페이지 확인 필요 (129 문의)
        

[2] 독거노인·장애인 응급안전안심서비스 (돌봄서비스)
▶ 서비스 설명: 안전의 사각지대에 있는 노인과 장애인이 응급상황을 인지하고 응급상황에 대처할 수 있도록 안전대책을 마련하여 지역사회 예방적 돌봄을 지원합니다.
▶ 누가 받을 수 있나요?: 노인 및 가족 문의: 보건복지부 콜센터129 한국사회보장정보원 사례관리정보부02-6360-5497
▶ 상세 확인 및 신청(URL): https://www.bokjiro.go.kr/ssis-tbu/twataa/wlfareInfo/moveTWAT52011M.do?wlfareInfoId=WLF00001093&wlfareInfoReldBztpCd=01
        


* 시나리오 3가지 테스트

In [ ]:
scenarios = [
    "매달 생활비가 부족해요. 집은 있어요.",
    "컴퓨터를 배우고 싶은데 어디로 가야 하나요?",
    "몸이 안 좋은데 혼자 있다가 쓰러질까 봐 걱정돼요."
]

for q in scenarios:
    print(f"\n🚀 시나리오 테스트: {q}")
    print(rag_recommend_v3(q))
    print("="*60)

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🚀 시나리오 테스트: 매달 생활비가 부족해요. 집은 있어요.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💬 "안녕하세요, 어르신님! 오늘 정말 행복하셨어요? 우리 가족에게 있어 소중한 존재로 여기며, 항상 함께해 주세요. 저희들은 여러분의 필요를 도와드릴 수 있는 다양한 지원책을 제공하겠습니다. 우리가 제공하는 모든 서비스를 최대한 활용해서 재정적 어려움을 극복하도록 노력하겠습니다. 여러분과 함께하는 시간이 되시길 바랍니다!"

[1] 요보호가정 지원사업 (기타지원)
▶ 서비스 설명: 장애인 월 15~25만원의 생활비(1년)- 아동 월 5~7만원의 학습비(1년)- 노인 연간 200만원 내외의 생활비
▶ 누가 받을 수 있나요?: 대전시 거주 1인이상 장애인 포함, 4인가족으로 저소득장애인 가정(국민기초생활수급권자 및 타 단체 지원대상 가정 제외)- 서울 강동구 거주 저소득 가정 초등학생(주민등록상)- 서
▶ 상세 확인 및 신청(URL): 공식 홈페이지 확인 필요 (129 문의)
        

[2] 주택담보노후연금보증 (경제지원)
▶ 서비스 설명: 노후생활에 어려움을 겪는 노인에 대해 보유하고 있는 주택을 담보로 매월 일정금액의 대출금을 연금형식으로 지급하여 안정적인 노후 생활을 지원합니다.
▶ 누가 받을 수 있나요?: 노인 및 가족 문의: 한국주택금융공사1688-8114
▶ 상세 확인 및 신청(URL): https://www.bokjiro.go.kr/ssis-tbu/twataa/wlfareInfo/moveTWAT52011M.do?wlfareInfoId=WLF00001108&wlfareInfoReldBztpCd=01
        

🚀 시나리오 테스트: 컴퓨터를 배우고 싶은데 어디로 가야 하나요?


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💬 "안녕하세요, 어르신님! 저는 노인 복지 전문가가 되어 여러분과 함께 즐거운 시간을 보낼 수 있도록 도와드리겠습니다. 여러분의 필요를 이해하고 필요한 모든 지원을 제공하겠습니다."

[1] [구립증산정보도서관] 어르신을 위한 디지털 특강 - AI로 통하는 세상 (디지털교육)
▶ 서비스 설명: 배우고 체험할 수 있는 무료 프로그램이에요. 장소: 수색동주민센터 3층 제1강의실
▶ 누가 받을 수 있나요?: 구글 계정을 가지고 있는 어르신 10명 문의: 02-307-6030
▶ 상세 확인 및 신청(URL): https://culture.seoul.go.kr/culture/culture/cultureEvent/view.do?cultcode=154527&menuNo=200011
        

[2] [서울도서관] 시니어디지털금융교육 : 넌 은행가니? 난 스마트폰으로 한다! (디지털교육)
▶ 서비스 설명: 배우고 체험할 수 있는 무료 프로그램이에요. 장소: 서울도서관 4층 사서교육장
▶ 누가 받을 수 있나요?: 50세 이상 시니어 서울도서관 회원 20명 문의: 02-2133-0263
▶ 상세 확인 및 신청(URL): https://culture.seoul.go.kr/culture/culture/cultureEvent/view.do?cultcode=154287&menuNo=200011
        

🚀 시나리오 테스트: 몸이 안 좋은데 혼자 있다가 쓰러질까 봐 걱정돼요.
💬 "안녕하세요, 어르신님! 저는 노인 복지 전문가로, 도와드리기 위해 이 자리를 마련했습니다. 건강 상태를 잘 모니터링해 주시고, 필요한 경우 도움을 줄 수 있도록 하겠습니다. 편안함을 느낄 수 있는 환경을 조성하여 안전하고 편안한 시간을 보내실 수 있도록 최선을 다하겠습니다."

[1] 산재근로자 합병증 등 예방관리 (돌봄서비스)
▶ 서비스 설명: 산재요양 종결 후 상병 및 장해의 특성으로 인하여 발생하는 합병증 등 예방관리를 지원합니다.
▶ 누가 받을 수 있나요?: 노인 및 가족 문의: 근로

> 시나리오 3은 임베딩 모델의 유사도 판단에 따라 의료 지원이 우선 추천되었으나, 향후 키워드 가중치 설정을 통해 응급 서비스를 우선 배치할 예정.